# ASSIGNMENT – SENTIMENT ANALYSIS USING NLP (IMDb DATASET)

# 1. Data Understanding

In [2]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
# checking number of rows and columns
print("Shape of dataset:", df.shape)

# printing column names
print("\nColumns:")
print(df.columns)

# checking class distribution
print("\nClass Distribution:")
print(df['sentiment'].value_counts())

Shape of dataset: (50000, 2)

Columns:
Index(['review', 'sentiment'], dtype='object')

Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


### Data Understanding

The dataset is loaded successfully and contains 50,000 samples across two columns.

* The main feature used is **"review"** (the text content).
* The target column is **"sentiment"** (the label).

The dataset has two classes:

* **positive**
* **negative**

From the class distribution, we can see that the dataset is **perfectly balanced**, meaning both classes have exactly 25,000 samples each. This helps in building a reliable machine learning model without class bias.

# 2. NLP Preprocessing

In [3]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import pandas as pd 

nltk.download('stopwords')

# loading stopwords
stop_words = set(stopwords.words('english'))

# using stemming
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data... 
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
def preprocess_text(text):

    # handle missing values
    if pd.isna(text):
        return ""

    # convert to lowercase
    text = text.lower()

    # remove HTML tags (important for IMDb dataset)
    text = re.sub(r'<.*?>', '', text)

    # remove urls
    text = re.sub(r'http\S+', '', text)

    # remove special characters and numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # tokenization
    words = text.split()

    # remove stopwords
    words = [w for w in words if w not in stop_words]

    # stemming
    words = [stemmer.stem(w) for w in words]

    return " ".join(words)

In [5]:
df['processed_text'] = df['review'].apply(preprocess_text)

df[['review', 'processed_text']].head()

,review,processed_text
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod youll hook...
1,A wonderful little production. The...,wonder littl product film techniqu unassum ol...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer week...
3,Basically there's a family where a little boy ...,basic there famili littl boy jake think there...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun fil...


### NLP Preprocessing

In this step, the raw text reviews are cleaned and normalized. 

The following techniques are applied:

* **Lowercase Conversion**: Standardizing all text to lowercase to ensure consistency.
* **HTML Tag Removal**: IMDb reviews often contain `<br />` tags which add noise.
* **URL & Special Character Removal**: Removing non-alphabetic noise like web links and punctuation.
* **Stopword Removal**: Filtering out common words (the, is, in) that don't carry significant sentiment.
* **Stemming**: Reducing words to their root form (e.g., "wonderful" to "wonder") to reduce vocabulary size.

This process significantly reduces the complexity of the text data while retaining the core sentiment meaning.

# 3. Feature Engineering

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

X = df['processed_text']
y = df['sentiment']

bow = CountVectorizer()
X_bow = bow.fit_transform(X)

print("BoW Shape:", X_bow.shape)

BoW Shape: (50000, 101230)


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(X)

print("TF-IDF Shape:", X_tfidf.shape)

TF-IDF Shape: (50000, 101230)


### Feature Engineering

To build a machine learning model, text must be converted into numerical vectors. Two primary methods are compared here:

* **Bag of Words (BoW)**: Simply counts the frequency of each word in the vocabulary across documents.
* **TF-IDF (Term Frequency-Inverse Document Frequency)**: A statistical measure used to evaluate how important a word is to a document in a collection. It penalizes very frequent words across all documents and boosts unique, meaningful words.

Both methods resulted in a vocabulary of approximately **101,230 unique tokens** for this dataset. TF-IDF is generally preferred for large text datasets as it better identifies key descriptive terms.

# 4. Model Building

In [8]:
from sklearn.model_selection import train_test_split

# splitting data into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)

# Decision Tree
dt = DecisionTreeClassifier(max_depth=20)
dt.fit(X_train, y_train)

print("Models trained successfully.")

Models trained successfully.


### Model Building

We trained three different machine learning architectures to compare their effectiveness on sentiment analysis using TF-IDF features:

1. **Logistic Regression**: A linear model often very effective for high-dimensional sparse text data.
2. **Naive Bayes (Multinomial)**: A probabilistic classifier based on Bayes' theorem, widely used as a baseline for NLP.
3. **Decision Tree**: A non-linear model that splits data based on feature importance. We limited the depth here to prevent extreme overfitting.

The dataset was split into **80% training** and **20% testing** sets to ensure objective evaluation.

# 5. Model Evaluation

In [10]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_lr = lr.predict(X_test)
print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression
Accuracy: 0.8935
              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      4961
    positive       0.89      0.90      0.90      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [11]:
y_pred_nb = nb.predict(X_test)
print("Naive Bayes")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Naive Bayes
Accuracy: 0.8652
              precision    recall  f1-score   support

    negative       0.86      0.87      0.86      4961
    positive       0.87      0.86      0.87      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



In [12]:
y_pred_dt = dt.predict(X_test)
print("Decision Tree")
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

Decision Tree
Accuracy: 0.7324
              precision    recall  f1-score   support

    negative       0.79      0.63      0.70      4961
    positive       0.69      0.83      0.76      5039

    accuracy                           0.73     10000
   macro avg       0.74      0.73      0.73     10000
weighted avg       0.74      0.73      0.73     10000



### Model Evaluation

We evaluated each model based on Accuracy, Precision, Recall, and F1-Score:

* **Logistic Regression** achieved the highest accuracy of **~89.35%**. It shows very balanced performance for both positive and negative classes.
* **Naive Bayes** performed well with **~86.52%** accuracy. It remains a very efficient choice for quick labeling.
* **Decision Tree** struggled more on this sparse data, achieving **~73.24%**. It showed a bias towards labels with more training variance when depth is restricted.

Overall, Logistic Regression is the clear winner for this specific NLP task.

# 6. Comparison & Insights

### Comparison & Insights

* **TF-IDF is superior to BoW** as it handles common word noise more intelligently by reducing the weight of words that appear in every review.
* **Logistic Regression leads overall** because text classification data is often linearly separable in high-dimensional vector space (like 100k+ features from TF-IDF).
* **Naive Bayes is useful for performance** as it trains almost instantly and delivers competitive results with minimal memory overhead.
* **Decision Trees are less effective for high-dimensional NLP** because they tend to overfit on specific words and fail to capture the cumulative sentiment from thousands of small word-signals.

# 7. Conclusion

### Conclusion

This project successfully implemented a complete NLP pipeline for Sentiment Analysis on the IMDb Movie Review dataset.

We progressed from raw HTML-laden text to a cleaned, tokenized, and stemmed set of features. By converting these features into numerical TF-IDF vectors, we were able to train multiple machine learning models.

**Final Recommendation**: For sentiment analysis tasks involving thousands of features, **Logistic Regression paired with TF-IDF Vectorization** provides the best balance of speed, accuracy, and interpretability.